# RQ2: Is the distribution of delay causes statistically associated with airline and state?

## Motivation
Different airlines/states have different prop-
erties which affect how well they can run on time. Some
are financial, geographic, infrastructural, or human, but all
are important to stakeholders. By looking for patterns in
the data by grouping, we hope to discover which prop-
erties have the largest effect for each category, allowing
for anticipation of issues, and perhaps giving direction
for improvement to each respective group. For example,
if the colder states happen to be more caused by human
error than weather conditions, this may inform decision
making, which would initially feel counterintuitive, but is
well supported by the data.

## Proposed Methodology
After cleaning the dataset and
standardizing delay cause categories, we will group flights
by airline and state. We will test the hypothesis that delay
cause probability distributions are independent of airline
and state. We will test this theory using chi-square tests
of independence and effect size measures (e.g., Cram´er’s
V) to quantify the strength of any observed relationships.
For cases where the group is too small, we will employ
hierarchical grouping, treating airline or state as group-level
effects. This model will allow us to estimate the probability
of each delay cause given a specific airline or state while
controlling variables such as climate and airport traffic
volume. The resulting probability distributions will be used
both for statistical analysis and for prediction to estimate the
most likely delay causes for a given subgroup

In [14]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np

In [15]:
# Import CSV in as Pandas data frame
path = "../data/flight-delay-dataset-20182022/raw/"
files = [
    "Flights_2018_1.csv",
    "Flights_2018_2.csv",
    "Flights_2018_3.csv",
    "Flights_2018_4.csv",
    "Flights_2018_5.csv",
    "Flights_2018_6.csv",
    "Flights_2018_7.csv",
    "Flights_2018_8.csv",
    "Flights_2018_9.csv",
    "Flights_2018_10.csv",
    "Flights_2018_11.csv",
    "Flights_2018_12.csv",

    "Flights_2019_1.csv",
    "Flights_2019_2.csv",
    "Flights_2019_3.csv",
    "Flights_2019_4.csv",
    "Flights_2019_5.csv",
    "Flights_2019_6.csv",
    "Flights_2019_7.csv",
    "Flights_2019_8.csv",
    "Flights_2019_9.csv",
    "Flights_2019_10.csv",
    "Flights_2019_11.csv",
    "Flights_2019_12.csv",

    "Flights_2020_1.csv",
    "Flights_2020_2.csv",
    "Flights_2020_3.csv",
    "Flights_2020_4.csv",
    "Flights_2020_5.csv",
    "Flights_2020_6.csv",
    "Flights_2020_7.csv",
    "Flights_2020_8.csv",
    "Flights_2020_9.csv",
    "Flights_2020_10.csv",
    "Flights_2020_11.csv",
    "Flights_2020_12.csv",

    "Flights_2021_1.csv",
    "Flights_2021_2.csv",
    "Flights_2021_3.csv",
    "Flights_2021_4.csv",
    "Flights_2021_5.csv",
    "Flights_2021_6.csv",
    "Flights_2021_7.csv",
    "Flights_2021_8.csv",
    "Flights_2021_9.csv",
    "Flights_2021_10.csv",
    "Flights_2021_11.csv",
    "Flights_2021_12.csv",

    "Flights_2022_1.csv",
    "Flights_2022_2.csv",
    "Flights_2022_3.csv",
    "Flights_2022_4.csv",
    "Flights_2022_5.csv",
    "Flights_2022_6.csv",
    "Flights_2022_7.csv"
]
cols = ["Operating_Airline ",
        "OriginState",
        "CarrierDelay",
        "WeatherDelay",
        "NASDelay",
        "SecurityDelay",
        "LateAircraftDelay"]

In [16]:
# Extract all useful columns and merge all data into one data frame
df_flight_data = pd.DataFrame(columns=cols)

for file in files:
    print("Loading " + file)
    df = pd.read_csv(path + file, usecols=cols)
    print("appending dataframe")
    df_flight_data = pd.concat([df_flight_data, df], ignore_index=True)
    del df
print("data loaded")

Loading Flights_2018_1.csv
appending dataframe
Loading Flights_2018_2.csv


C:\Users\nswco\AppData\Local\Temp\ipykernel_47080\3875293463.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_flight_data = pd.concat([df_flight_data, df], ignore_index=True)


appending dataframe
Loading Flights_2018_3.csv
appending dataframe
Loading Flights_2018_4.csv
appending dataframe
Loading Flights_2018_5.csv
appending dataframe
Loading Flights_2018_6.csv
appending dataframe
Loading Flights_2018_7.csv
appending dataframe
Loading Flights_2018_8.csv
appending dataframe
Loading Flights_2018_9.csv
appending dataframe
Loading Flights_2018_10.csv
appending dataframe
Loading Flights_2018_11.csv
appending dataframe
Loading Flights_2018_12.csv
appending dataframe
Loading Flights_2019_1.csv
appending dataframe
Loading Flights_2019_2.csv
appending dataframe
Loading Flights_2019_3.csv
appending dataframe
Loading Flights_2019_4.csv
appending dataframe
Loading Flights_2019_5.csv
appending dataframe
Loading Flights_2019_6.csv
appending dataframe
Loading Flights_2019_7.csv
appending dataframe
Loading Flights_2019_8.csv
appending dataframe
Loading Flights_2019_9.csv
appending dataframe
Loading Flights_2019_10.csv
appending dataframe
Loading Flights_2019_11.csv
appendin

In [17]:
df_flight_data = df_flight_data.dropna()

delay_cols = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay"
]

# Keep only rows with actual delay causes
df_flight_data = df_flight_data[
    df_flight_data[delay_cols].sum(axis=1) > 0
]

# Create categorical delay cause
df_flight_data["DelayCause"] = (
    df_flight_data[delay_cols]
    .idxmax(axis=1)
    .str.replace("Delay", "")
)

In [18]:
# Top airlines
top_airlines = df_flight_data["Operating_Airline "].value_counts().nlargest(28).index
df_flight_data["AirlineGrouped"] = df_flight_data["Operating_Airline "].where(
    df_flight_data["Operating_Airline "].isin(top_airlines), "Other"
)

# Top states
top_states = df_flight_data["OriginState"].value_counts().nlargest(52).index
df_flight_data["StateGrouped"] = df_flight_data["OriginState"].where(
    df_flight_data["OriginState"].isin(top_states), "Other"
)

In [19]:
airline_table = pd.crosstab(
    df_flight_data["AirlineGrouped"],
    df_flight_data["DelayCause"]
)

state_table = pd.crosstab(
    df_flight_data["StateGrouped"],
    df_flight_data["DelayCause"]
)

airline_table

DelayCause,Carrier,LateAircraft,NAS,Security,Weather
AirlineGrouped,,,,,
9E,36122,45478,39741,104,5261
9K,93,39,15,0,1
AA,186289,196486,160030,1874,21010
AS,38734,44453,61618,1142,2895
AX,10806,17142,10496,1,1135
B6,98762,97673,72061,1000,5380
C5,17283,24787,20979,9,2193
CP,7610,12601,7088,27,427
DL,155673,114301,136594,800,15757


In [20]:
chi2_airline, p_airline, _, exp_airline = chi2_contingency(airline_table)
chi2_state, p_state, _, exp_state = chi2_contingency(state_table)

print("Airline p-value:", p_airline)
print("State p-value:", p_state)

Airline p-value: 0.0
State p-value: 0.0


In [21]:
def cramers_v(chi2, table):
    n = table.values.sum()
    r, k = table.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

cv_airline = cramers_v(chi2_airline, airline_table)
cv_state = cramers_v(chi2_state, state_table)

print("Cramer's V (Airline):", cv_airline)
print("Cramer's V (State):", cv_state)

Cramer's V (Airline): 0.12629124998847457
Cramer's V (State): 0.08388699059119802


In [ ]:
airline_probs = airline_table.div(airline_table.sum(axis=1), axis=0)
state_probs = state_table.div(state_table.sum(axis=1), axis=0)

airline_probs